In [1]:
from utils.experiment_utils import get_all_experiments_info, load_best_model
import torch
import os
import hydra
from omegaconf import DictConfig, OmegaConf

import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader

import ot

from datasets.lineage_tracing import LTSeqDataset

from geomloss import SamplesLoss

from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, r2_score

from utils.eval_utils import compute_mmd_distance, compute_sw_distance

from sklearn.neighbors import NearestNeighbors

/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [2]:
lts = LTSeqDataset(seed=42)

loading cached adata from ./data/processed/adata_pca_50.h5ad  !!
loading cached clone sets from ./data/processed!
splitting 1218 clones into 609 train and 609 test


In [3]:
configs = get_all_experiments_info('/orcd/home/002/gokulg/orcd/scratch/CoupledDistributionEmbeddings/outputs/', False)

cfgs = [c for c in configs if 'Bidirectional' in c['dataset']]

cfgs

[{'name': 'lineage_semisupervised_f063e8fc90fcaba0bb7279a7725dcb9b',
  'dir': '/orcd/home/002/gokulg/orcd/scratch/CoupledDistributionEmbeddings/outputs/lineage_semisupervised_f063e8fc90fcaba0bb7279a7725dcb9b',
  'config': {'dataset': {'_target_': 'datasets.lineage_tracing.LTSeqDatasetUnstructuredBidirectional', 'set_size': 100, 'min_cells': 3, 'data_shape': [50], 'root': '/orcd/home/002/gokulg/orcd/scratch/CoupledDistributionEmbeddings/data', 'seed': '${seed}'}, 'encoder': {'_target_': 'encoder.encoders.DistributionEncoderGNN', 'in_dim': '${dataset.data_shape[0]}', 'latent_dim': '${experiment.latent_dim}', 'hidden_dim': '${experiment.hidden_dim}', 'set_size': '${experiment.set_size}', 'layers': 2, 'fc_layers': 2}, 'model': {'_target_': 'layers.MLP', 'in_dims': [50, 128, 128], 'hidden_dim': 512, 'out_dim': 50, 'layers': 4}, 'coupling': {'_target_': 'types.NoneType'}, 'generator': {'_target_': 'generator.direct.DirectGenerator', 'model': '${model}', 'loss_type': 'mmd', 'noise_dim': '${ex

In [5]:
def load_model(cfg, path, device):
    enc = hydra.utils.instantiate(cfg['encoder'])
    gen = hydra.utils.instantiate(cfg['generator'])
    state = load_best_model(path)
    enc.load_state_dict(state['encoder_state_dict'])
    gen.load_state_dict(state['generator_state_dict'])
    enc.eval()
    gen.eval()
    enc.to(device)
    gen.to(device)
    return enc, gen

In [8]:
results = {
    'energy' : [],
    'mmd' : [],
    'swd' : []
}

energy = SamplesLoss('energy')

device = 'cuda'

for model in cfgs:
    print(f"Evaluating model: {model['name']}")
    encoder, generator = load_model(model['config'], model['dir'], 'cuda')
    

    with torch.no_grad():

        z_x_train = encoder(lts.train_srcs.to(device))
        z_y_train = encoder(lts.train_tgts.to(device))
        z_x_test = encoder(lts.test_srcs.to(device))
        z_y_test = encoder(lts.test_tgts.to(device))

    X_train = z_x_train.cpu().numpy()
    Y_train = z_y_train.cpu().numpy()
    X_test = z_x_test.cpu().numpy()
    Y_test = z_y_test.cpu().numpy()

    # fit ridge regression
    alphas = np.logspace(-6, 6, 25)
    ridge = RidgeCV(alphas=alphas, cv=5, scoring='neg_mean_squared_error')
    ridge.fit(X_train, Y_train)
    
    # predict target embeddings
    Y_pred_test = ridge.predict(X_test)
    mse = mean_squared_error(Y_test, Y_pred_test)
    r2 = r2_score(Y_test, Y_pred_test, multioutput='variance_weighted')
    
    # semi-supervised with predicted embeddings
    y_hat_semi = generator.sample(
        lts.test_srcs.to(device).reshape(-1, 50),
        z_x_test.to(device),
        torch.tensor(Y_pred_test, dtype=torch.float32).to(device)
    ).reshape(lts.test_tgts.shape)

    energy_errors = energy(lts.test_tgts.to(device), y_hat_semi.to(device))
    mmd_errors = compute_mmd_distance(lts.test_tgts.to(device), y_hat_semi.to(device))
    swd_errors = compute_sw_distance(lts.test_tgts.to(device), y_hat_semi.to(device), n_projections=100)

    for energy_error, mmd_error, swd_error in zip(energy_errors, mmd_errors, swd_errors):
        results['energy'].append(energy_error.item())
        results['mmd'].append(mmd_error.item())
        results['swd'].append(swd_error.item())


Evaluating model: lineage_semisupervised_f063e8fc90fcaba0bb7279a7725dcb9b


In [9]:
results_df = pd.DataFrame(results)
print(results_df.mean())
print(results_df.std()/np.sqrt(len(results_df)))

energy    4.231215
mmd       8.462452
swd       1.740155
dtype: float64
energy    0.114719
mmd       0.229439
swd       0.024045
dtype: float64
